In [0]:
#Read Bronze Tables
bronze_customers_df=spark.table("bronze_customers")
bronze_transactions_df=spark.table("bronze_transactions")

In [0]:
#View Schema
bronze_customers_df.printSchema()
bronze_transactions_df.printSchema()

In [0]:
#Remove Duplicate Customers
silver_customers_df=bronze_customers_df.dropDuplicates(
    ["customer_id"]
)

In [0]:
#Handle Null Values
from pyspark.sql.functions import col
silver_customers_df=silver_customers_df.filter(
    col("customer_id").isNotNull()
)

In [0]:
#Standarize Text Columns
from pyspark.sql.functions import upper
silver_customers_df = silver_customers_df.withColumn(
    "country",
    upper(col("country"))
)

silver_customers_df = silver_customers_df.withColumn(
    "account_type",
    upper(col("account_type"))
)

In [0]:
#Clean Transaction Data
silver_transactions_df=bronze_transactions_df.filter(
    col("transaction_amount") > 0
)

In [0]:
#Remove Duplicate Transactions
silver_transactions_df=silver_transactions_df.dropDuplicates(
    ["transaction_id"]
)

In [0]:
#Add Business Logic Column
from pyspark.sql.functions import when
silver_transactions_df=silver_transactions_df.withColumn(
    "amount_category",
    when(col("transaction_amount")<1000,"LOW")
    .when(col("transaction_amount")>=1000,"HIGH")
    .otherwise("HIGH")
)

In [0]:
#Join Customers + Transaction Data
silver_joined_df = silver_transactions_df.join(
    silver_customers_df,
    on ="customer_id",
    how="left"
)

In [0]:
#Verify Joined Data
display(silver_joined_df)

In [0]:
#Save Silver Tables
silver_customers_df.write \
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("silver_customers")

silver_transactions_df.write \
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("silver_transactions")

silver_joined_df_clean = silver_joined_df.drop(silver_joined_df.columns[-1])
silver_joined_df_clean.write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("silver_finance")

In [0]:
#Verify Tables
display(
    spark.sql("""
    SELECT * FROM silver_finance
    LIMIT 20
    """)
)